# 00_05: HuggingFace Model Configuration & Customization

**Learning Objectives:**
- Understand the anatomy of `config.json` and how it defines model architecture
- Modify configurations to create custom-sized models
- Build randomly initialized models from scratch using configs
- Perform model surgery: freeze layers, swap heads, extract sub-models
- Estimate model memory requirements from parameter counts

**Prerequisites:** Notebooks 00_01 through 00_04. No GPU required.

## Prerequisites

| Requirement | Specification |
|-------------|---------------|
| **CPU** | Any modern CPU |
| **RAM** | 4GB minimum |
| **Storage** | ~500MB for model downloads |
| **GPU** | Not required |
| **Internet** | Required for first-time model downloads |

## Expected Behaviors

### What You'll See
- Model configs are simple JSON dictionaries that fully specify an architecture
- Changing a config field (e.g., `num_hidden_layers`) changes the model's parameter count
- Models created from configs have random weights -- they need training before they're useful
- Freezing layers reduces trainable parameters without changing model behavior

### First Run
- Downloads DistilBERT (~268MB), DistilGPT-2 (~82MB), and T5-small (~242MB)
- Creates several tiny custom models in memory (no downloads needed)

### Common Observations
- Smaller configs = fewer parameters = less memory = faster inference
- Random-weight models produce garbage output -- this is expected
- Most parameters live in embeddings and attention layers

## Overview

Every HuggingFace model is defined by two things:

1. **Configuration** (`config.json`) -- the architectural blueprint: how many layers, how many attention heads, what hidden size, vocabulary size, etc.
2. **Weights** (`.safetensors` or `.bin`) -- the learned parameters that make the model useful

When you call `AutoModel.from_pretrained("distilbert-base-uncased")`, HuggingFace downloads both the config and the weights. But you can also work with configs independently:

- **Inspect** a config to understand what architecture you're running
- **Modify** a config to change the architecture (e.g., fewer layers for faster inference)
- **Create from scratch** to build a randomly initialized model for training
- **Perform surgery** on loaded models to freeze layers, swap heads, or extract components

Understanding configs gives you control over the models you use, rather than treating them as black boxes.

## Setup and Installation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import random
import transformers
from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    DistilBertConfig,
    GPT2Config,
)
import matplotlib.pyplot as plt
import pandas as pd
import sys

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Part 1: Anatomy of config.json

A model's config is a flat dictionary of hyperparameters that fully define the architecture. Let's load one and examine every field.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
config = AutoConfig.from_pretrained(MODEL_NAME)

# Display all config fields organized by category
print(f'=== Config for {MODEL_NAME} ===')
print(f'Model type: {config.model_type}')
print(f'Architectures: {config.architectures}\n')

architecture_fields = {
    'hidden_size': 'Dimension of hidden states (embedding width)',
    'n_layers': 'Number of transformer layers',
    'n_heads': 'Number of attention heads per layer',
    'dim': 'Alias for hidden_size in DistilBERT',
    'hidden_dim': 'Feed-forward intermediate dimension',
    'vocab_size': 'Number of tokens in vocabulary',
    'max_position_embeddings': 'Maximum sequence length',
}

rows: list[dict[str, str]] = []
for field, description in architecture_fields.items():
    value = getattr(config, field, 'N/A')
    rows.append({'Field': field, 'Value': str(value), 'Description': description})

pd.DataFrame(rows)

### How Config Fields Map to Architecture

The relationship between config fields and model structure is direct:

$$\text{head\_dim} = \frac{\text{hidden\_size}}{\text{num\_heads}}$$

$$\text{attention\_params\_per\_layer} \approx 4 \times \text{hidden\_size}^2$$

$$\text{ffn\_params\_per\_layer} \approx 2 \times \text{hidden\_size} \times \text{intermediate\_size}$$

More layers and wider hidden sizes mean more parameters, more memory, and better capacity -- but also slower inference.

In [ ]:
def compare_configs(model_names: list[str]) -> pd.DataFrame:
    """Compare configurations across multiple model families.

    Args:
        model_names: List of HuggingFace model identifiers.

    Returns:
        DataFrame comparing key config fields.
    """
    field_aliases = {
        'hidden_size': ['hidden_size', 'n_embd', 'd_model', 'dim'],
        'num_hidden_layers': ['num_hidden_layers', 'n_layer', 'num_layers', 'n_layers'],
        'num_attention_heads': ['num_attention_heads', 'n_head', 'num_heads', 'n_heads'],
        'intermediate_size': ['intermediate_size', 'n_inner', 'd_ff', 'hidden_dim'],
        'vocab_size': ['vocab_size'],
        'max_position_embeddings': ['max_position_embeddings', 'n_positions', 'n_ctx'],
    }

    data: dict[str, list[str]] = {'Field': list(field_aliases.keys())}
    for name in model_names:
        cfg = AutoConfig.from_pretrained(name)
        values: list[str] = []
        for field, aliases in field_aliases.items():
            value = 'N/A'
            for alias in aliases:
                if hasattr(cfg, alias):
                    value = str(getattr(cfg, alias))
                    break
            values.append(value)
        short_name = name.split('/')[-1]
        data[short_name] = values

    return pd.DataFrame(data)


print('=== Config Comparison Across Model Families ===')
compare_configs(['distilbert-base-uncased', 'distilgpt2', 't5-small'])

Notice the differences: DistilBERT is an encoder-only model, GPT-2 is decoder-only, and T5 is encoder-decoder. Each has different vocabulary sizes and position limits, and uses different naming conventions for the same concepts (`dim` vs `n_embd` vs `d_model`). Despite these differences, `AutoConfig` handles them all uniformly.

## Part 2: Modifying Configurations

You can modify any config field to create a custom architecture. This is useful for:
- Creating smaller models for experimentation (fewer layers, smaller hidden size)
- Understanding how architecture choices affect parameter count
- Building distilled or compressed model variants

Let's create a "TinyBERT" by shrinking DistilBERT's config.

In [ ]:
# Load the original config
original_config = AutoConfig.from_pretrained('distilbert-base-uncased')

# Create a tiny variant by modifying fields
tiny_config = DistilBertConfig(
    vocab_size=original_config.vocab_size,  # Keep same vocabulary
    max_position_embeddings=128,             # Shorter sequences
    dim=128,                                 # Much smaller hidden size (was 768)
    n_layers=2,                              # Fewer layers (was 6)
    n_heads=2,                               # Fewer heads (was 12)
    hidden_dim=256,                          # Smaller FFN (was 3072)
)

print('=== Original vs Tiny Config ===')
comparison_fields = ['dim', 'n_layers', 'n_heads', 'hidden_dim', 'max_position_embeddings']
config_comparison = pd.DataFrame({
    'Field': comparison_fields,
    'Original (distilbert-base)': [getattr(original_config, f) for f in comparison_fields],
    'Tiny (custom)': [getattr(tiny_config, f) for f in comparison_fields],
    'Reduction': [
        f'{getattr(original_config, f) / getattr(tiny_config, f):.1f}x'
        for f in comparison_fields
    ],
})
config_comparison

In [ ]:
def count_parameters(model: torch.nn.Module) -> dict[str, int]:
    """Count total and trainable parameters in a model.

    Args:
        model: A PyTorch model.

    Returns:
        Dictionary with total and trainable parameter counts.
    """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {'total': total, 'trainable': trainable}


# Build both models and compare
original_model = AutoModel.from_pretrained('distilbert-base-uncased')
tiny_model = AutoModel.from_config(tiny_config)

orig_params = count_parameters(original_model)
tiny_params = count_parameters(tiny_model)

print('=== Parameter Count Comparison ===')
param_comparison = pd.DataFrame({
    'Model': ['DistilBERT (original)', 'TinyBERT (custom)'],
    'Total Parameters': [f'{orig_params["total"]:,}', f'{tiny_params["total"]:,}'],
    'Memory (FP32)': [
        f'{orig_params["total"] * 4 / 1e6:.1f} MB',
        f'{tiny_params["total"] * 4 / 1e6:.1f} MB',
    ],
    'Memory (FP16)': [
        f'{orig_params["total"] * 2 / 1e6:.1f} MB',
        f'{tiny_params["total"] * 2 / 1e6:.1f} MB',
    ],
})
param_comparison

In [ ]:
# The relationship between hidden_size, num_heads, and head_dim
print('=== Attention Head Dimensions ===')
for name, cfg in [('Original', original_config), ('Tiny', tiny_config)]:
    head_dim = cfg.dim // cfg.n_heads
    print(f'{name}: hidden_size={cfg.dim}, num_heads={cfg.n_heads}, head_dim={head_dim}')

print(f'\nRule: hidden_size must be divisible by num_heads.')
print(f'  head_dim = hidden_size / num_heads')
print(f'  Each head attends to a {tiny_config.dim // tiny_config.n_heads}-dimensional subspace.')

## Part 3: Creating Models from Scratch

When you call `AutoModel.from_config(config)` instead of `AutoModel.from_pretrained(name)`, you get a model with the right architecture but **randomly initialized weights**. The model hasn't learned anything -- it produces random outputs. This is the starting point for training a model from scratch.

Let's verify this by comparing pretrained vs random model outputs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
test_input = tokenizer('Hello world!', return_tensors='pt')

# Pretrained model -- meaningful outputs
pretrained = AutoModel.from_pretrained('distilbert-base-uncased').to(device)
pretrained.eval()
with torch.no_grad():
    pretrained_out = pretrained(**test_input.to(device))

# Random model -- garbage outputs
random_config = AutoConfig.from_pretrained('distilbert-base-uncased')
random_model = AutoModel.from_config(random_config).to(device)
random_model.eval()
with torch.no_grad():
    random_out = random_model(**test_input.to(device))

print('=== Pretrained vs Random Model Outputs ===')
pretrained_cls = pretrained_out.last_hidden_state[0, 0, :8]
random_cls = random_out.last_hidden_state[0, 0, :8]

output_comparison = pd.DataFrame({
    'Property': ['Output shape', 'Mean', 'Std', 'First 4 values'],
    'Pretrained': [
        str(tuple(pretrained_out.last_hidden_state.shape)),
        f'{pretrained_out.last_hidden_state.mean():.4f}',
        f'{pretrained_out.last_hidden_state.std():.4f}',
        str([f'{v:.4f}' for v in pretrained_cls[:4].tolist()]),
    ],
    'Random (from_config)': [
        str(tuple(random_out.last_hidden_state.shape)),
        f'{random_out.last_hidden_state.mean():.4f}',
        f'{random_out.last_hidden_state.std():.4f}',
        str([f'{v:.4f}' for v in random_cls[:4].tolist()]),
    ],
})
output_comparison

Both models have the same architecture and output shape, but the pretrained model produces structured, meaningful representations while the random model outputs noise. This is the difference that training makes -- the architecture is just a container.

In [ ]:
# Create a tiny GPT-2 from scratch
tiny_gpt2_config = GPT2Config(
    vocab_size=50257,       # Standard GPT-2 vocabulary
    n_positions=256,        # Short context (was 1024)
    n_embd=64,              # Tiny hidden size (was 768)
    n_layer=2,              # Minimal layers (was 12)
    n_head=2,               # Minimal heads (was 12)
    n_inner=128,            # Small FFN (was 3072)
)

tiny_gpt2 = AutoModelForCausalLM.from_config(tiny_gpt2_config).to(device)
tiny_gpt2.eval()

gpt2_tokenizer = AutoTokenizer.from_pretrained('distilgpt2')

# Generate text (will be random gibberish -- model is untrained)
input_ids = gpt2_tokenizer.encode('The meaning of life is', return_tensors='pt').to(device)
with torch.no_grad():
    output_ids = tiny_gpt2.generate(
        input_ids,
        max_new_tokens=20,
        do_sample=True,
        temperature=1.0,
    )

generated = gpt2_tokenizer.decode(output_ids[0], skip_special_tokens=True)

tiny_params = count_parameters(tiny_gpt2)
print(f'=== Tiny GPT-2 from Scratch ===')
print(f'Parameters: {tiny_params["total"]:,} ({tiny_params["total"] * 4 / 1e6:.1f} MB in FP32)')
print(f'\nGenerated text (random -- untrained):')
print(f'  "{generated}"')
print(f'\nThis is expected! The model has random weights and needs training.')

## Part 4: Model Surgery

"Model surgery" refers to modifying a loaded model's structure -- freezing parameters so they don't update during training, replacing output heads for new tasks, or extracting specific layers. These techniques are fundamental to **transfer learning** and **fine-tuning**, which you'll use extensively in later notebooks.

### Inspecting Model Layers

Before operating on a model, you need to know its layer structure. PyTorch provides `named_parameters()` and `named_modules()` for this.

In [ ]:
model = AutoModel.from_pretrained('distilbert-base-uncased').to(device)


def inspect_model_layers(
    model: torch.nn.Module,
    max_depth: int = 2,
) -> pd.DataFrame:
    """List model layers with their types and parameter counts.

    Args:
        model: A PyTorch model.
        max_depth: Maximum nesting depth to display.

    Returns:
        DataFrame with layer names, types, and parameter counts.
    """
    rows: list[dict[str, str]] = []
    for name, module in model.named_modules():
        depth = name.count('.')
        if depth > max_depth or name == '':
            continue
        params = sum(p.numel() for p in module.parameters(recurse=False))
        if params > 0:
            rows.append({
                'Layer': name,
                'Type': type(module).__name__,
                'Own Parameters': f'{params:,}',
            })
    return pd.DataFrame(rows)


print('=== DistilBERT Layer Structure (depth=2) ===')
inspect_model_layers(model, max_depth=2)

### Freezing Layers

Freezing a layer means setting `requires_grad = False` on its parameters. Frozen layers still compute forward passes, but their weights won't be updated during backpropagation. This is the core technique behind transfer learning: keep the pretrained knowledge frozen and only train a new output head.

Common freezing strategies:
- **Freeze all, train head only** -- fastest, least flexible
- **Freeze embeddings + lower layers** -- balance of speed and adaptability
- **Freeze nothing (full fine-tuning)** -- most flexible, most expensive

In [ ]:
def freeze_layers(
    model: torch.nn.Module,
    freeze_pattern: str,
) -> dict[str, int]:
    """Freeze model parameters matching a name pattern.

    Args:
        model: A PyTorch model.
        freeze_pattern: String pattern -- parameters whose names contain this are frozen.

    Returns:
        Dictionary with total, trainable, and frozen parameter counts.
    """
    for name, param in model.named_parameters():
        if freeze_pattern in name:
            param.requires_grad = False

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        'total': total,
        'trainable': trainable,
        'frozen': total - trainable,
    }


def unfreeze_all(model: torch.nn.Module) -> None:
    """Unfreeze all parameters in a model.

    Args:
        model: A PyTorch model.
    """
    for param in model.parameters():
        param.requires_grad = True


# Load a classification model for surgery demos
cls_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased-finetuned-sst-2-english'
).to(device)

# Strategy 1: Freeze everything except the classification head
unfreeze_all(cls_model)
stats_head_only = freeze_layers(cls_model, 'distilbert')

# Strategy 2: Freeze only embeddings
unfreeze_all(cls_model)
stats_emb_only = freeze_layers(cls_model, 'embeddings')

# Strategy 3: Full fine-tuning (nothing frozen)
unfreeze_all(cls_model)
stats_full = count_parameters(cls_model)
stats_full['frozen'] = 0

print('=== Freezing Strategy Comparison ===')
freeze_df = pd.DataFrame({
    'Strategy': [
        'Full fine-tuning (nothing frozen)',
        'Freeze embeddings only',
        'Freeze base (train head only)',
    ],
    'Trainable': [
        f'{stats_full["trainable"]:,}',
        f'{stats_emb_only["trainable"]:,}',
        f'{stats_head_only["trainable"]:,}',
    ],
    'Frozen': [
        f'{stats_full["frozen"]:,}',
        f'{stats_emb_only["frozen"]:,}',
        f'{stats_head_only["frozen"]:,}',
    ],
    'Trainable %': [
        f'{stats_full["trainable"] / stats_full["total"]:.1%}',
        f'{stats_emb_only["trainable"] / stats_emb_only["total"]:.1%}',
        f'{stats_head_only["trainable"] / stats_head_only["total"]:.1%}',
    ],
    'Training Speed': ['Slowest', 'Medium', 'Fastest'],
})
freeze_df

### Swapping Classification Heads

A common transfer learning pattern: take a model pretrained on one task (e.g., sentiment analysis with 2 classes) and replace its output head for a different task (e.g., topic classification with 5 classes). The pretrained base model retains its learned language understanding, and you only need to train the new head.

In [ ]:
def swap_classification_head(
    model: AutoModelForSequenceClassification,
    num_new_labels: int,
) -> AutoModelForSequenceClassification:
    """Replace the classification head with a new one for a different number of classes.

    Args:
        model: A pretrained sequence classification model.
        num_new_labels: Number of classes for the new task.

    Returns:
        The model with a new randomly initialized classification head.
    """
    hidden_size = model.config.hidden_size

    # Replace the classifier head
    model.classifier = torch.nn.Linear(hidden_size, num_new_labels)
    model.num_labels = num_new_labels
    model.config.num_labels = num_new_labels

    # Initialize the new head
    torch.nn.init.xavier_uniform_(model.classifier.weight)
    torch.nn.init.zeros_(model.classifier.bias)

    return model


# Original model: 2-class sentiment
print(f'Original: {cls_model.config.num_labels} classes (sentiment: positive/negative)')
print(f'Classifier: {cls_model.classifier}')

# Swap to 5-class topic classification
NUM_TOPIC_CLASSES = 5
cls_model = swap_classification_head(cls_model, NUM_TOPIC_CLASSES)
print(f'\nAfter swap: {cls_model.config.num_labels} classes (topics)')
print(f'Classifier: {cls_model.classifier}')

# Verify it works
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
test_input = tokenizer('This is a test sentence.', return_tensors='pt').to(device)
cls_model.eval()
with torch.no_grad():
    output = cls_model(**test_input)
print(f'\nOutput logits shape: {output.logits.shape} -- one score per class')
print(f'Logits: {output.logits[0].tolist()}')
print(f'(Random values -- the new head needs training)')

### Extracting Sub-Models

Sometimes you only need part of a model -- the embedding layer for feature extraction, or specific transformer layers for analysis. You can access these directly through the model's module hierarchy.

In [ ]:
# Access the base DistilBERT inside the classification model
base = cls_model.distilbert
embeddings = base.embeddings
transformer_layers = base.transformer.layer

print('=== Model Components ===')
print(f'Base model: {type(base).__name__}')
print(f'Embeddings: {type(embeddings).__name__}')
print(f'Number of transformer layers: {len(transformer_layers)}')

# Get embeddings only (useful for feature extraction)
test_input = tokenizer('Extract my features!', return_tensors='pt').to(device)

with torch.no_grad():
    # Get just the word embeddings (before transformer layers)
    word_embeds = embeddings.word_embeddings(test_input['input_ids'])
    print(f'\nWord embeddings shape: {word_embeds.shape}')
    print(f'  -> Raw token representations before any attention')

    # Get full model output (after all transformer layers)
    full_output = base(**test_input)
    print(f'Full output shape: {full_output.last_hidden_state.shape}')
    print(f'  -> Contextualized representations after all attention layers')

## Part 5: Parameter Counting & Memory Estimation

Before running a model, it's useful to know how much memory it will require. The formula is straightforward:

$$\text{Memory (bytes)} = \text{num\_parameters} \times \text{bytes\_per\_parameter}$$

| Precision | Bytes per Parameter | Typical Use |
|-----------|--------------------|-----------|
| FP32 (float32) | 4 | Training (default) |
| FP16 (float16) | 2 | Mixed-precision training, inference |
| INT8 | 1 | Quantized inference |
| INT4 | 0.5 | Aggressive quantization |

Note: actual GPU memory usage is higher than weight storage because of activation tensors, optimizer states, and gradients.

In [ ]:
def parameter_breakdown(
    model: torch.nn.Module,
    top_n: int = 10,
) -> pd.DataFrame:
    """Break down parameters by layer group.

    Args:
        model: A PyTorch model.
        top_n: Number of top layer groups to display.

    Returns:
        DataFrame with parameter counts per layer group.
    """
    groups: dict[str, int] = {}
    for name, param in model.named_parameters():
        parts = name.split('.')
        group = '.'.join(parts[:3]) if len(parts) >= 3 else name
        groups[group] = groups.get(group, 0) + param.numel()

    total = sum(groups.values())
    sorted_groups = sorted(groups.items(), key=lambda x: x[1], reverse=True)

    rows: list[dict[str, str]] = []
    for name, count in sorted_groups[:top_n]:
        rows.append({
            'Layer Group': name,
            'Parameters': f'{count:,}',
            'Percentage': f'{count / total:.1%}',
            'Memory (FP32)': f'{count * 4 / 1e6:.1f} MB',
        })

    rows.append({
        'Layer Group': 'TOTAL',
        'Parameters': f'{total:,}',
        'Percentage': '100.0%',
        'Memory (FP32)': f'{total * 4 / 1e6:.1f} MB',
    })

    return pd.DataFrame(rows)


print('=== DistilBERT Parameter Breakdown ===')
base_model = AutoModel.from_pretrained('distilbert-base-uncased')
parameter_breakdown(base_model)

In [ ]:
def visualize_parameter_distribution(
    model: torch.nn.Module,
    model_name: str,
) -> None:
    """Create a bar chart showing parameter distribution across layer types.

    Args:
        model: A PyTorch model.
        model_name: Display name for the chart title.
    """
    categories: dict[str, int] = {
        'Embeddings': 0,
        'Attention': 0,
        'Feed-Forward': 0,
        'Layer Norm': 0,
        'Other': 0,
    }

    for name, param in model.named_parameters():
        n = param.numel()
        name_lower = name.lower()
        if 'embedding' in name_lower:
            categories['Embeddings'] += n
        elif 'attention' in name_lower or 'attn' in name_lower:
            categories['Attention'] += n
        elif any(k in name_lower for k in ['intermediate', 'ffn', 'lin1', 'lin2']):
            categories['Feed-Forward'] += n
        elif 'norm' in name_lower or 'ln' in name_lower:
            categories['Layer Norm'] += n
        else:
            categories['Other'] += n

    # Remove empty categories
    categories = {k: v for k, v in categories.items() if v > 0}
    total = sum(categories.values())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#607D8B']
    bars = axes[0].bar(
        categories.keys(),
        [v / 1e6 for v in categories.values()],
        color=colors[:len(categories)],
    )
    axes[0].set_ylabel('Parameters (millions)')
    axes[0].set_title(f'{model_name} -- Parameters by Type')
    for bar, count in zip(bars, categories.values()):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            f'{count / total:.0%}',
            ha='center', va='bottom', fontsize=10,
        )

    # Pie chart
    axes[1].pie(
        categories.values(),
        labels=categories.keys(),
        autopct='%1.1f%%',
        colors=colors[:len(categories)],
        startangle=90,
    )
    axes[1].set_title(f'{model_name} -- Parameter Distribution')

    plt.tight_layout()
    plt.show()


visualize_parameter_distribution(base_model, 'DistilBERT')

In [ ]:
def estimate_memory(model_names: list[str]) -> pd.DataFrame:
    """Estimate memory requirements for multiple models at different precisions.

    Args:
        model_names: List of HuggingFace model identifiers.

    Returns:
        DataFrame with memory estimates.
    """
    rows: list[dict[str, str]] = []
    for name in model_names:
        model = AutoModel.from_pretrained(name)
        total = sum(p.numel() for p in model.parameters())
        del model  # Free memory

        rows.append({
            'Model': name.split('/')[-1],
            'Parameters': f'{total / 1e6:.1f}M',
            'FP32 (4B)': f'{total * 4 / 1e9:.2f} GB',
            'FP16 (2B)': f'{total * 2 / 1e9:.2f} GB',
            'INT8 (1B)': f'{total * 1 / 1e9:.2f} GB',
            'INT4 (0.5B)': f'{total * 0.5 / 1e9:.2f} GB',
        })

    return pd.DataFrame(rows)


print('=== Memory Estimation Across Models ===')
estimate_memory(['distilbert-base-uncased', 'distilgpt2', 't5-small'])

Remember: these are **weight-only** memory estimates. During training, you also need memory for:
- **Gradients**: same size as weights (1x)
- **Optimizer states**: 2x weights for Adam (momentum + variance)
- **Activations**: varies by batch size and sequence length

A rough rule of thumb: training requires **4-8x** the weight-only memory. Inference requires only the weights plus a small activation buffer.

## Part 6: Config Best Practices

A few important rules to follow when working with model configurations.

In [ ]:
# Best Practice 1: Always save config with model
import tempfile
import os

save_dir = os.path.join(tempfile.gettempdir(), 'config_demo')
os.makedirs(save_dir, exist_ok=True)

# Save a model -- config is saved automatically
tiny_model_save = AutoModel.from_config(tiny_config)
tiny_model_save.save_pretrained(save_dir)

saved_files = sorted(os.listdir(save_dir))
print('=== Files Saved with Model ===')
for f in saved_files:
    size = os.path.getsize(os.path.join(save_dir, f))
    print(f'  {f:35s} {size / 1024:.1f} KB')
print(f'\nconfig.json is always saved -- it tells AutoModel which class to load.')

# Clean up
import shutil
shutil.rmtree(save_dir, ignore_errors=True)

In [ ]:
# Best Practice 2: Verify config compatibility
print('=== Common Config Pitfalls ===')
pitfalls = pd.DataFrame({
    'Pitfall': [
        'Mismatched vocab_size and tokenizer',
        'hidden_size not divisible by num_heads',
        'max_position_embeddings too small',
        'Loading weights with wrong config',
    ],
    'What Happens': [
        'Index out of range error on unknown tokens',
        'RuntimeError in attention layer',
        'Cannot process sequences longer than limit',
        'Shape mismatch error on load',
    ],
    'How to Avoid': [
        'Always use same model name for config and tokenizer',
        'Verify: hidden_size % num_heads == 0',
        'Set max_length in tokenizer to match config',
        'Use from_pretrained() -- it loads config automatically',
    ],
})
pitfalls

## Exercises

1. **Config exploration**: Load configs for `bert-base-uncased` and `bert-large-uncased`. Calculate the parameter ratio between them using only config fields (hidden_size, num_layers, vocab_size). Verify by loading both models and counting parameters.

2. **Custom architecture**: Create a DistilBERT config with `dim=256`, `n_layers=4`, `n_heads=4`. Build a model from it, run a forward pass, and calculate its memory footprint. How does it compare to the standard DistilBERT?

3. **Selective freezing**: Load `distilbert-base-uncased` and freeze only the first 3 transformer layers (keep layers 4-6 and the classifier trainable). Count trainable vs frozen parameters. Hint: transformer layers are indexed in `model.distilbert.transformer.layer[i]`.

4. **Head swap pipeline**: Load a sentiment model, swap its head for 4 classes, freeze the base, and run a forward pass. Verify the output shape is `(batch_size, 4)`.

## Key Takeaways

- **`config.json` is the blueprint** -- it fully specifies model architecture through fields like `hidden_size`, `num_hidden_layers`, and `num_attention_heads`
- **Modify configs to create custom architectures** -- shrink models for experimentation by reducing layers, hidden size, or heads
- **`AutoModel.from_config()`** creates randomly initialized models -- same architecture, no pretrained knowledge, ready for training from scratch
- **Model surgery enables transfer learning** -- freeze pretrained layers, swap classification heads, and extract sub-models for new tasks
- **Memory scales linearly with parameters** -- FP32 uses 4 bytes/param, FP16 halves it, and training needs 4-8x the weight-only memory

## Next Steps & Resources

**Next section**: [01_01 -- Text Generation](../01_nlp/01_01_nlp_text_generation.ipynb) -- apply your foundational knowledge to build a text generation system.

**Resources:**
- [Configuration Documentation](https://huggingface.co/docs/transformers/main_classes/configuration) -- Complete AutoConfig API reference
- [Model Documentation](https://huggingface.co/docs/transformers/main_classes/model) -- AutoModel and model loading
- [Transfer Learning Guide](https://huggingface.co/docs/transformers/training) -- Fine-tuning and freezing strategies
- [Model Memory Calculator](https://huggingface.co/docs/accelerate/usage_guides/model_size_estimator) -- Accelerate's memory estimator tool
- [safetensors Format](https://huggingface.co/docs/safetensors/) -- Modern weight serialization format